## Data Cleaning & Reduction

In [1]:
import numpy as np
import pandas as pd
import random
import math

import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.preprocessing.image import ImageDataGenerator, array_to_img, img_to_array, load_img

from sklearn.preprocessing import LabelEncoder
from sklearn import ensemble
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import matplotlib.pyplot as plt
import seaborn as sns

import os
import shutil
import pathlib
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore", category=UserWarning) #used to supress the tf version warning. 

2024-04-14 13:55:42.144391: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Kaggle Dataset - Cleaned Using 1st Letter

In [2]:
wiki_df = pd.read_csv('/Users/jimxu/Downloads/en.tsv',sep = '\t') 

# Cutting down Wiki_df

grouped = wiki_df.groupby(wiki_df['page_name'].str[0])
reduced_data = []
    
for name, group in grouped:
    num_to_keep = int(len(group) * 0.077203134262) # keeping 500,000 rows
    reduced_group = group.sample(n=num_to_keep, random_state=42) # randomize selections
    reduced_data.append(reduced_group)
    
reduced_wiki_df = pd.concat(reduced_data)
reduced_wiki_df.shape


(499895, 10)

In [3]:
reduced_wiki_df.head(10)

,page_id,page_name,wikirank_quality,norm_len,norm_refs,norm_img,norm_sec,norm_reflen,norm_authors,flawtemps
32203,64486,! (disambiguation),24.2702,3.45,0.00,15.38,28.57,0.00,98.21,0.0
5777868,60793860,"$50,000 Reward",30.7239,13.00,11.02,15.38,42.86,86.91,15.18,0.0
3075944,27116683,$ (Mark Sultan album),21.9583,6.47,3.39,38.46,14.29,53.76,22.32,1.0
3100545,27412213,$h*! My Dad Says,58.1454,76.61,39.83,7.69,71.43,53.31,157.14,0.0
4198736,40415293,&pictures,20.0401,4.59,1.69,15.38,21.43,37.86,39.29,0.0
1581935,11104921,'Aql,27.5300,11.66,8.47,0.00,35.71,74.51,34.82,0.0
3708872,34774248,'Ajde Jano,40.1941,31.72,33.90,7.69,42.86,109.59,25.00,0.0
1970717,15081803,'In Wrong' Wright,13.2319,3.72,0.85,7.69,28.57,23.39,15.18,0.0
6177834,66247775,'Round the World with Les Baxter,16.8802,5.25,3.39,7.69,14.29,66.20,4.46,0.0
6399883,69300751,'Aunofo Havea Funaki,27.3074,10.98,9.32,15.38,35.71,87.09,5.36,0.0


In [4]:
# Further extracting 10% from each category.

grouped = reduced_wiki_df.groupby(reduced_wiki_df['page_name'].str[0])
reduced_data_2 = []
    
for name, group in grouped:
    num_to_keep = int(len(group) * 0.10) # keeping 500,000 rows
    reduced_group_2 = group.sample(n=num_to_keep, random_state=42) # randomize selections
    reduced_data_2.append(reduced_group_2)
    
further_reduced_wiki_df = pd.concat(reduced_data_2)
further_reduced_wiki_df.shape

(49950, 10)

### Figshare Dataset - Cleaned Using Each Category

In [7]:
figshare_df = pd.read_csv('/Users/jimxu/Downloads/topics_all_wikipedia_articles_202012.tsv', sep = '\t')
figshare_df['urls']='en.wikipedia.org/wiki/?curid=' + figshare_df['pid'].astype(str)

figshare_df=figshare_df.drop(figshare_df[figshare_df.wiki_db!='enwiki'].index)

max_column_names = []

for index, row in figshare_df.iterrows():
    max_value = None
    max_column_name = None
    for col_name, col_value in row.items():
        if pd.api.types.is_float_dtype(figshare_df[col_name]):
            if max_value is None or col_value > max_value:
                max_value = col_value
                max_column_name = col_name
    max_column_names.append(max_column_name)  

figshare_df['categories'] = max_column_names

category_counts = figshare_df['categories'].value_counts()
desired_counts = (category_counts * 0.01469339296).astype(int)
    
truncated_dfs = []
    
for category, count in desired_counts.items():
    category_df = figshare_df[figshare_df['categories'] == category]
    truncated_category_df = category_df.head(count)
    truncated_dfs.append(truncated_category_df)
    
truncated_figshare_df = pd.concat(truncated_dfs)

truncated_figshare_df.shape

(91608, 70)